# 05. Model Merging: SLERP, Task Arithmetic, TIES, DARE, and Evolutionary Merging

- **Category:** 15_compression_peft_and_speculative_inference / 02_adaptive_kv_speculative_and_merging
- **Lineage & Citations:** Shoemake (SIGGRAPH 1985, *Animating Rotation with Quaternion Curves*); Ilharco et al. (ICLR 2023, *Editing Models with Task Arithmetic*); Yadav et al. (NeurIPS 2023, *TIES-Merging: Resolving Interference When Merging Models*); Yu et al. (ICML 2024, *DARE: Language Models are Super Mario*); Akiba et al. (2024, *Evolutionary Model Merging*).
- **Purpose:** Implement zero-shot parameter composition in weight space: spherical linear interpolation (SLERP), linear task arithmetic, interference-proof TIES-Merging, and extreme stochastic pruning with DARE.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [Tree Speculative Medusa Eagle GoT and Jacobi](04_tree_speculative_medusa_eagle_got_and_jacobi.ipynb) $\to$ `🔗 Downstream Syntheses:` [PEFT LoRA Rslora Loraplus and DoRA](../03_parameter_efficient_finetuning_peft/01_peft_lora_rslora_loraplus_and_dora.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Never merge models trained from disparate pretraining lineages or distinct architectures (e.g. LLaMA with Mistral); parameter coordinate spaces are not aligned and produce catastrophic weight incoherence. Do not merge $>2$ tasks with unregularized task arithmetic without TIES or DARE sign resolution.

## Mathematical Foundations & Governing Equations

### 1. Task Vector & Linear Arithmetic
$$\tau_m = \theta_m - \theta_{\text{base}}, \quad \theta_{\text{merged}} = \theta_{\text{base}} + \sum_{m=1}^M \lambda_m \tau_m$$

### 2. Spherical Linear Interpolation (SLERP)
$$\omega = \arccos\left(\frac{v_0 \cdot v_1}{\|v_0\| \|v_1\|}\right), \quad \text{SLERP}(v_0, v_1, t) = \frac{\sin((1-t)\omega)}{\sin(\omega)} v_0 + \frac{\sin(t\omega)}{\sin(\omega)} v_1$$

### 3. TIES-Merging (Trimming, Electing Sign, Disjoint Merge)
$$\text{Trim}: \hat{\tau}_{m} = \text{top-}k(|\tau_m|), \quad \gamma = \text{sgn}\left(\sum_m \hat{\tau}_m\right), \quad \bar{\tau}_j = \frac{1}{|A_j|} \sum_{m \in A_j} \hat{\tau}_{m, j}$$

### 4. DARE (Drop And REscale)
$$\tilde{\tau} = \frac{1}{1 - p} \cdot \text{Bernoulli}(1 - p) \odot \tau$$

### Complexity
Weight-space merging operates entirely in $\mathcal{O}(P)$ parameter space with zero training data, zero forward passes, and zero GPU memory for gradients.

In [ ]:
import torch
import safetensors.torch
from pathlib import Path
import core

env = core.setup_env(seed=42)
CKPT_REL = Path("weights/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.safetensors")
ckpt_path = next(p / CKPT_REL for p in (Path.cwd(), *Path.cwd().parents) if (p / CKPT_REL).exists())
weights = safetensors.torch.load_file(ckpt_path)
print(f"Loaded {len(weights)} golden tensors on {env.device}")

In [ ]:
def linear_anchor(x, w, b):
    return x @ w + b

def slerp_merge(v0, v1, t, eps=1e-6):
    orig_shape = v0.shape
    v0_f, v1_f = v0.flatten(), v1.flatten()
    n0, n1 = torch.norm(v0_f), torch.norm(v1_f)
    if n0 < eps or n1 < eps:
        return (1.0 - t) * v0 + t * v1
    u0, u1 = v0_f / n0, v1_f / n1
    dot = torch.sum(u0 * u1).clamp(-1.0, 1.0)
    theta = torch.acos(dot)
    if torch.sin(theta) < eps or torch.abs(theta) < eps:
        return (1.0 - t) * v0 + t * v1
    s = torch.sin(theta)
    res = (torch.sin((1.0 - t) * theta) / s) * v0_f + (torch.sin(t * theta) / s) * v1_f
    return res.reshape(orig_shape)

def task_arithmetic_merge(base, deltas, task_weights):
    merged = base.clone()
    for k in range(deltas.shape[0]):
        merged += task_weights[k] * deltas[k]
    return merged

def ties_merge(base, deltas, trim_ratio=0.5, scale=0.5):
    num_tasks, d1, d2 = deltas.shape
    tot = d1 * d2
    keep_k = max(1, int(round(tot * (1.0 - trim_ratio))))
    trimmed = torch.zeros_like(deltas)
    for m in range(num_tasks):
        flat = deltas[m].flatten()
        _, top_idx = torch.topk(torch.abs(flat), k=keep_k, largest=True)
        trimmed_f = torch.zeros_like(flat)
        trimmed_f[top_idx] = flat[top_idx]
        trimmed[m] = trimmed_f.reshape(d1, d2)
    elected = torch.sign(torch.sum(trimmed, dim=0))
    merged_delta = torch.zeros((d1, d2), dtype=deltas.dtype, device=deltas.device)
    counts = torch.zeros((d1, d2), dtype=deltas.dtype, device=deltas.device)
    for m in range(num_tasks):
        agree = (torch.sign(trimmed[m]) == elected) & (trimmed[m] != 0.0)
        merged_delta += torch.where(agree, trimmed[m], torch.zeros_like(trimmed[m]))
        counts += agree.to(deltas.dtype)
    avg = torch.where(counts > 0, merged_delta / counts, torch.zeros_like(merged_delta))
    return base + scale * avg

def dare_rescale(delta, mask, drop_prob=0.5):
    return delta * mask * (1.0 / (1.0 - drop_prob))

In [ ]:
y = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-6

s_out = slerp_merge(weights["v0"], weights["v1"], 0.3)
assert float(torch.max(torch.abs(s_out - weights["slerp_out"]))) < 1e-5

a_out = task_arithmetic_merge(weights["base_w"], weights["deltas"], weights["task_weights"])
assert float(torch.max(torch.abs(a_out - weights["arithmetic_out"]))) < 1e-5

t_out = ties_merge(weights["base_w"], weights["deltas"], 0.5, 0.6)
assert float(torch.max(torch.abs(t_out - weights["ties_out"]))) < 1e-5

d_out = dare_rescale(weights["deltas"][0], weights["dare_mask"], 0.5)
assert float(torch.max(torch.abs(d_out - weights["dare_out"]))) < 1e-5
print("All 5 model merging algorithms match golden checkpoints (< 1e-5)!")

In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_slerp = benchmark_fn(lambda: slerp_merge(weights["v0"], weights["v1"], 0.3), iters=100, name="SLERP Merge")
b_ties = benchmark_fn(lambda: ties_merge(weights["base_w"], weights["deltas"], 0.5, 0.6), iters=100, name="TIES-Merge (3 tasks)")
print(format_benchmark_table([b_slerp, b_ties]))

norm_v0 = torch.norm(weights["v0"]).item()
norm_slerp = torch.norm(s_out).item()
lerp_naive = 0.7 * weights["v0"] + 0.3 * weights["v1"]
norm_lerp = torch.norm(lerp_naive).item()
print(f"Norm Preservation: Base={norm_v0:.3f}, SLERP={norm_slerp:.3f}, LERP={norm_lerp:.3f} (LERP collapses by {(1 - norm_lerp/norm_v0)*100:.1f}%!)")

In [ ]:
# Pathological Misuse: Naive Task Arithmetic suffers catastrophic sign cancellation
base = torch.zeros(1, 4)
delta_math = torch.tensor([[+1.0, +1.0, +1.0, +1.0]])
delta_code = torch.tensor([[-1.0, +1.0, -1.0, +1.0]])
deltas_conflict = torch.stack([delta_math, delta_code])

# Naive sum: position 0 and 2 cancel to 0.0 (+1.0 + -1.0 = 0.0) -> loses both skills!
naive_sum = delta_math + delta_code
assert naive_sum[0, 0].item() == 0.0, "Naive sum suffers complete feature cancellation!"

# TIES-Merge: detects non-conflicting agreement at pos 1 and 3, preventing silent cancellation
ties_res = ties_merge(base, deltas_conflict, trim_ratio=0.0, scale=1.0)
assert ties_res[0, 1].item() == 1.0, "TIES preserves full magnitude on agreed coordinates!"
print("Verified: TIES-Merging resolves parameter sign collisions that break naive task arithmetic.")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Zero-Compute Multi-Task Fusion**: Weight-space merging combines distinct expert capabilities without retraining or GPU memory for gradients.
2. **Norm Preservation via SLERP**: Interpolating on the spherical manifold prevents activation norm collapse common in naive linear averaging.
3. **TIES Eliminates Sign Interference**: Filtering conflicting parameter signs preserves skill fidelity when assembling multi-expert models.

---
### Cross-Domain Generalisation Matrix
- **Computer Vision:** SLERP blends latent vectors in StyleGAN and Stable Diffusion without image blur.
- **Robotics:** Merging distinct motor policy networks (walking + stair climbing) via task vectors.
- **Federated Learning:** TIES-Merging aggregates heterogeneous client updates while rejecting Byzantine parameter drift.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.md](../../../markdown/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.md)
- **PyTorch 2.x Module:** [05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.py](../../../pytorch2/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.py)
- **Functional JAX Module:** [05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.py](../../../jax/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.py)
- **Native Rust Candle Track:** [05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.rs](../../../candle/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.rs)
- **Hardware Kernel (CPU / WGSL):** [05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.rs](../../../kernels/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.rs)
- **Golden SafeTensors:** [05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.safetensors](../../../weights/15_compression_peft_and_speculative_inference/02_adaptive_kv_speculative_and_merging/05_model_merging_slerp_task_arithmetic_ties_dare_and_evolutionary.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 04. Tree Speculative Medusa Eagle GoT and Jacobi](04_tree_speculative_medusa_eagle_got_and_jacobi.ipynb) | [Master Index](../../../README.md#15_compression_peft_and_speculative_inference) | [Next: 01. PEFT LoRA Rslora Loraplus and DoRA](../03_parameter_efficient_finetuning_peft/01_peft_lora_rslora_loraplus_and_dora.ipynb) &rarr;